In [4]:
import pandas as pd
from pathlib import Path
df = pd.read_csv("../tabular_4/tabular_4.csv")

print(df.shape)
print(df.columns.tolist())



(100000, 17)
['year', 'gender', 'age', 'location', 'race:AfricanAmerican', 'race:Asian', 'race:Caucasian', 'race:Hispanic', 'race:Other', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'hbA1c_level', 'blood_glucose_level', 'diabetes', 'clinical_notes']


In [ ]:
missing_summary = df.isnull().sum()

# Print columns with missing values (display only those with missing values)
missing_cols = missing_summary[missing_summary > 0]
print("Total number of missing columns:", len(missing_cols))
display(missing_cols)

# Proportion of missing data
missing_ratio = (df.isnull().mean() * 100).round(2)
print("\n Proportion of missing values（%）Top 10 columns：")
display(missing_ratio.sort_values(ascending=False).head(10))

Total number of missing columns: 0


Series([], dtype: int64)


 Proportion of missing values（%）Top 10 columns：


year                   0.0
hypertension           0.0
diabetes               0.0
blood_glucose_level    0.0
hbA1c_level            0.0
bmi                    0.0
smoking_history        0.0
heart_disease          0.0
race:Other             0.0
gender                 0.0
dtype: float64

In [ ]:
import numpy as np

# Path settings
in_path  = Path("../tabular_4/tabular_4.csv")
out_dir  = Path("../tabular_4")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "tabular_4_aligned_mask_templated.csv"

# Template field
TEMPLATE_FEATURES = [
    "Age", "Sex", "BMI", "GenHlth",
    "HighBP", "DiffWalk", "HighChol", "HeartDiseaseorAttack"
]
LABEL_COL = "Diabetes_012"

# Mapping
COL_ALIASES = {
    # label
    "diabetes": "Diabetes_012",
    "diabetes_012": "Diabetes_012",

    # features
    "age": "Age",

    "sex": "Sex",
    "gender": "Sex",

    "bmi": "BMI",
    "body_mass_index": "BMI",

    "genhlth": "GenHlth",
    "general_health": "GenHlth",
    "generalhlth": "GenHlth",

    "highbp": "HighBP",
    "high_blood_pressure": "HighBP",
    "hypertension": "HighBP",

    "diffwalk": "DiffWalk",
    "difficulty_walking": "DiffWalk",
    "difficultywalking": "DiffWalk",

    "highchol": "HighChol",
    "high_cholesterol": "HighChol",
    "cholesterol_high": "HighChol",

    "heartdiseaseorattack": "HeartDiseaseorAttack",
    "heart_disease": "HeartDiseaseorAttack",
    "mi": "HeartDiseaseorAttack",
    "myocardial_infarction": "HeartDiseaseorAttack",
    "chd": "HeartDiseaseorAttack",
}

def canon(s: str) -> str:
    return "".join(ch for ch in s.lower() if ch.isalnum())

# Coercion functions
def to_num(x):
    try:
        if isinstance(x, str):
            x = x.strip()
            if x == "": 
                return np.nan
        return float(x)
    except:
        return np.nan

def to_int(x):
    y = to_num(x)
    return np.nan if pd.isna(y) else int(round(y))

def coerce_age(s: pd.Series):
    # Age
    vals = s.map(to_num)
    mask = (~vals.isna()) & (vals >= 0) & (vals <= 100)
    vals = vals.where(mask, 0).astype(np.float32)  
    return vals, mask.astype("int8")

def coerce_sex(s: pd.Series):
    # Sex
    def map_sex(v):
        if v is None or (isinstance(v, float) and np.isnan(v)): return np.nan
        if isinstance(v, str):
            t = v.strip().lower()
            if t in ("male","m","man","1"):  return 1
            if t in ("female","f","woman","0"): return 0
            return np.nan
        
        n = to_num(v)
        if pd.isna(n): return np.nan
        if n in (0.0, 0, 1.0, 1): return int(n)
        return np.nan
    vals = s.map(map_sex)
    mask = vals.notna()
    vals = vals.fillna(0).astype(np.int64)
    return vals, mask.astype("int8")

def coerce_bmi(s: pd.Series):
    # BMI
    vals = s.map(to_num)
    mask = (~vals.isna()) & (vals >= 0) & (vals <= 80)
    vals = vals.where(mask, 0).round(2).astype(np.float32)
    return vals, mask.astype("int8")

def coerce_genhlth(s: pd.Series):
    # GenHlth
    def map_gen(v):
        if v is None or (isinstance(v, float) and np.isnan(v)): return np.nan
        if isinstance(v, str):
            t = v.strip().lower()
            table = {
                "excellent": 1, "very good": 2, "good": 3, "fair": 4, "poor": 5,
                "1":1,"2":2,"3":3,"4":4,"5":5
            }
            return table.get(t, np.nan)
        n = to_int(v)
        if n in (1,2,3,4,5): return n
        return np.nan
    vals = s.map(map_gen)
    mask = vals.notna()
    vals = vals.fillna(0).astype(np.int64)
    return vals, mask.astype("int8")

def coerce_binary(s: pd.Series):
    # HighBP / DiffWalk / HighChol / HeartDiseaseorAttack
    def map_bin(v):
        if v is None or (isinstance(v, float) and np.isnan(v)): return np.nan
        if isinstance(v, str):
            t = v.strip().lower()
            if t in ("1","yes","y","true","t"): return 1
            if t in ("0","no","n","false","f"): return 0
            return np.nan
        n = to_num(v)
        if pd.isna(n): return np.nan
        if n in (0.0, 0, 1.0, 1): return int(n)
        return np.nan
    vals = s.map(map_bin)
    mask = vals.notna()
    vals = vals.fillna(0).astype(np.int64)
    return vals, mask.astype("int8")

COERCE_FN = {
    "Age": coerce_age,
    "Sex": coerce_sex,
    "BMI": coerce_bmi,
    "GenHlth": coerce_genhlth,
    "HighBP": coerce_binary,
    "DiffWalk": coerce_binary,
    "HighChol": coerce_binary,
    "HeartDiseaseorAttack": coerce_binary,
}


df = pd.read_csv(in_path)

ds2tpl = {}
for col in df.columns:
    c = canon(col)
    if c in COL_ALIASES:
        ds2tpl[col] = COL_ALIASES[c]

aligned_feat = pd.DataFrame(index=df.index)
aligned_mask = pd.DataFrame(index=df.index)

for feat in TEMPLATE_FEATURES:
    src = None
    for k, v in ds2tpl.items():
        if v == feat:
            src = k
            break
    if src is not None:
        vals, m = COERCE_FN[feat](df[src])
    else:
        vals = pd.Series(0, index=df.index)
        m    = pd.Series(0, index=df.index, dtype="int8")
    aligned_feat[feat] = vals
    aligned_mask[f"{feat}_mask"] = m

# Processing the label column
label_src = None
for k, v in ds2tpl.items():
    if v == LABEL_COL:
        label_src = k
        break

if label_src is not None and label_src in df.columns:
    # Diabetes_012 
    def map_label(x):
        n = to_num(x)
        if pd.isna(n): return 0
        m = int(round(n))
        return m if m in (0,1,2) else 0
    y = df[label_src].map(map_label).astype(np.int64)
else:
    y = pd.Series(0, index=df.index, dtype=np.int64)

# Splicing:8+8+1
aligned_full = pd.concat([aligned_feat, aligned_mask.astype("int8")], axis=1)
aligned_full[LABEL_COL] = y

aligned_full.to_csv(out_path, index=False)
print("Saved:", out_path)
aligned_full.head()


Saved: ../tabular_4/tabular_4_aligned_mask_templated.csv


,Age,Sex,BMI,GenHlth,HighBP,DiffWalk,HighChol,HeartDiseaseorAttack,Age_mask,Sex_mask,BMI_mask,GenHlth_mask,HighBP_mask,DiffWalk_mask,HighChol_mask,HeartDiseaseorAttack_mask,Diabetes_012
0,32.0,0,27.320000,0,0,0,0,0,1,1,1,0,1,0,0,0,0
1,29.0,0,19.950001,0,0,0,0,0,1,1,1,0,1,0,0,0,0
2,18.0,1,23.760000,0,0,0,0,0,1,1,1,0,1,0,0,0,0
3,41.0,1,27.320000,0,0,0,0,0,1,1,1,0,1,0,0,0,0
4,52.0,0,23.750000,0,0,0,0,0,1,1,1,0,1,0,0,0,0


In [26]:
from sklearn.model_selection import train_test_split
import pandas as pd
from pathlib import Path

data_path = Path("../tabular_4/tabular_4_aligned_mask_templated.csv")
df = pd.read_csv(data_path)

# Parameter Settings
train_ratio = 0.7   
val_ratio   = 0.15  
test_ratio  = 0.15  
random_seed = 42    # Random seed for reproducibility

train_df, temp_df = train_test_split(df, test_size=(1 - train_ratio), random_state=random_seed, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=test_ratio / (test_ratio + val_ratio), random_state=random_seed)

out_dir = Path("../tabular_4")
train_path = out_dir / "tabular_4_train.csv"
val_path   = out_dir / "tabular_4_val.csv"
test_path  = out_dir / "tabular_4_test.csv"

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Training set: {train_df.shape}, Validation set: {val_df.shape}, Test set: {test_df.shape}")
print(f"Save path:\n- {train_path}\n- {val_path}\n- {test_path}")


Training set: (69999, 17), Validation set: (15000, 17), Test set: (15001, 17)
Save path:
- ../tabular_4/tabular_4_train.csv
- ../tabular_4/tabular_4_val.csv
- ../tabular_4/tabular_4_test.csv
